In [11]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

sys.path.append(os.path.abspath("../dataset and other libs"))
from cleaningcls import clean_cls
from ctf import CorrelationThresholdFilter
from pridict_thresh import pridict_thresh


In [12]:
# Load raw dataset
df = pd.read_csv('D:/Repos/Chrun-Pridictor/dataset and other libs/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Separate features (X) and target (y)
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print("\nTarget distribution in train set:")
print(y_train.value_counts(normalize=True).round(3))

Training set shape: (5634, 20)
Test set shape: (1409, 20)

Target distribution in train set:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


In [34]:

pipeline_dt = Pipeline([
    ('clean', clean_cls()),
    ('ctf', CorrelationThresholdFilter(threshold=0.14)),
    ('pridict_thresh', pridict_thresh(RandomForestClassifier(
            n_estimators=200,           # More trees for stability
            max_depth=8,             # Prevent hyper-specific splits
            class_weight='balanced',    # Handle churn class imbalance
            random_state=42), thresh=0.64))])    
                             # Your custom high-precision threshold
pipeline_dt.fit(X_train, y_train)
y_prob_dt = pipeline_dt.predict_proba(X_test)[:,1]
y_pred_dt = pipeline_dt.predict(X_test)
# y_pred_dt = (pipeline_dt.predict_proba(X_test)[: , 1] > 0.45).astype(int)

auc_dt = roc_auc_score(y_test, y_prob_dt)
print('=== Pipeline Evaluation ===')
print(f'ROC-AUC Score: {auc_dt:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_dt))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt))

=== Pipeline Evaluation ===
ROC-AUC Score: 0.8423

Confusion Matrix:
[[896 139]
 [138 236]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.87      0.87      1035
           1       0.63      0.63      0.63       374

    accuracy                           0.80      1409
   macro avg       0.75      0.75      0.75      1409
weighted avg       0.80      0.80      0.80      1409



In [21]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.arange(0.25, 0.66, 0.01)
for t in thresholds:
    y_pred = (y_prob_dt > t).astype(int)
    print(f"thresh={t:.2f} | P={precision_score(y_test,y_pred):.2f} | R={recall_score(y_test,y_pred):.2f} | F1={f1_score(y_test,y_pred):.2f}")


thresh=0.25 | P=0.41 | R=0.92 | F1=0.57
thresh=0.26 | P=0.42 | R=0.91 | F1=0.57
thresh=0.27 | P=0.42 | R=0.91 | F1=0.58
thresh=0.28 | P=0.43 | R=0.90 | F1=0.58
thresh=0.29 | P=0.43 | R=0.89 | F1=0.58
thresh=0.30 | P=0.43 | R=0.89 | F1=0.58
thresh=0.31 | P=0.44 | R=0.89 | F1=0.59
thresh=0.32 | P=0.45 | R=0.89 | F1=0.59
thresh=0.33 | P=0.45 | R=0.88 | F1=0.60
thresh=0.34 | P=0.45 | R=0.87 | F1=0.60
thresh=0.35 | P=0.46 | R=0.87 | F1=0.60
thresh=0.36 | P=0.46 | R=0.86 | F1=0.60
thresh=0.37 | P=0.47 | R=0.86 | F1=0.61
thresh=0.38 | P=0.47 | R=0.86 | F1=0.61
thresh=0.39 | P=0.48 | R=0.86 | F1=0.62
thresh=0.40 | P=0.48 | R=0.86 | F1=0.62
thresh=0.41 | P=0.49 | R=0.85 | F1=0.62
thresh=0.42 | P=0.49 | R=0.84 | F1=0.62
thresh=0.43 | P=0.50 | R=0.83 | F1=0.63
thresh=0.44 | P=0.50 | R=0.82 | F1=0.62
thresh=0.45 | P=0.51 | R=0.81 | F1=0.62
thresh=0.46 | P=0.51 | R=0.79 | F1=0.62
thresh=0.47 | P=0.52 | R=0.78 | F1=0.62
thresh=0.48 | P=0.52 | R=0.78 | F1=0.62
thresh=0.49 | P=0.53 | R=0.78 | F1=0.63


In [15]:
import pandas as pd

# 1. Get the trained model from your pipeline
model = pipeline_dt['pridict_thresh'].model

# 2. Get the feature importances (scores of how much the tree uses each feature)
importances = model.feature_importances_

# 3. Get the feature names from your pipeline's preprocessing steps

feature_names = pipeline_dt['ctf'].get_feature_names_out()


# 4. Combine them into a clean DataFrame and sort by most important
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# important_f = feature_importance_df[feature_importance_df['Importance'] > 0]
active_features = feature_importance_df

print("=== Features Used by the Decision Tree (Ranked by Importance) ===")
print(active_features.to_string(index=False))

=== Features Used by the Decision Tree (Ranked by Importance) ===
                              Feature  Importance
                               tenure    0.196779
                         TotalCharges    0.139810
                    Contract_Two year    0.116855
          InternetService_Fiber optic    0.100134
                       MonthlyCharges    0.091358
       PaymentMethod_Electronic check    0.066641
                    Contract_One year    0.044589
                      TechSupport_Yes    0.034703
                   OnlineSecurity_Yes    0.033021
      TechSupport_No internet service    0.018533
                   InternetService_No    0.018065
                     PaperlessBilling    0.018009
  StreamingMovies_No internet service    0.017699
     OnlineBackup_No internet service    0.016734
   OnlineSecurity_No internet service    0.016131
 DeviceProtection_No internet service    0.014600
      StreamingTV_No internet service    0.013792
                           Depende

In [16]:
feature_names

Index(['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PaperlessBilling',
       'MonthlyCharges', 'TotalCharges', 'InternetService_Fiber optic',
       'InternetService_No', 'OnlineSecurity_No internet service',
       'OnlineSecurity_Yes', 'OnlineBackup_No internet service',
       'DeviceProtection_No internet service',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service',
       'StreamingMovies_No internet service', 'Contract_One year',
       'Contract_Two year', 'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check'],
      dtype='object')

In [17]:

sample_output = pipeline_dt['clean'].transform(X_train.head(5))
sample_output = pipeline_dt['ctf'].transform(sample_output)
feature_names = list(sample_output.columns)

In [18]:
from trace_path import trace_customer_path
import trace_path

X_transformed = pipeline_dt['clean'].transform(X_train.head(1))
X_filtered = pipeline_dt['ctf'].transform(X_transformed)
trace_customer_path(pipeline_dt, X_filtered )

ℹ️  RandomForest detected — tracing through tree #0 of 200
Customer's Decision Path (Node IDs): [ 0  1  2  3 35 36 37 38 39]
Final Leaf Node: 39
Node 0 (Split Node):
  - Feature checked: 'Contract_Two year'
  - Tree rule: <= 0.5000
  - Customer's value: 0.0  →  Goes LEFT (<=)
----------------------------------------
Node 1 (Split Node):
  - Feature checked: 'InternetService_No'
  - Tree rule: <= 0.5000
  - Customer's value: 0.0  →  Goes LEFT (<=)
----------------------------------------
Node 2 (Split Node):
  - Feature checked: 'TechSupport_Yes'
  - Tree rule: <= 0.5000
  - Customer's value: 0.0  →  Goes LEFT (<=)
----------------------------------------
Node 3 (Split Node):
  - Feature checked: 'tenure'
  - Tree rule: <= 16.5000
  - Customer's value: 35.0  →  Goes RIGHT (>)
----------------------------------------
Node 35 (Split Node):
  - Feature checked: 'PaymentMethod_Electronic check'
  - Tree rule: <= 0.5000
  - Customer's value: 0.0  →  Goes LEFT (<=)
---------------------------

c:\Users\HomePC\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\HomePC\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(


In [19]:
print(feature_names)
cleaned_features = pipeline_dt.named_steps['clean'].transform(X_test.reset_index(drop=True))
filtered_features = pipeline_dt.named_steps['ctf'].transform(cleaned_features.reset_index(drop=True))
for i in range(filtered_features.shape[0]):
    print(filtered_features.iloc[i].to_dict())
    print('prediction: ', y_pred_dt[i])
    print('-'*20)

['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'DeviceProtection_No internet service', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingMovies_No internet service', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check']
{'SeniorCitizen': 0.0, 'Partner': 1.0, 'Dependents': 1.0, 'tenure': 72.0, 'PaperlessBilling': 1.0, 'MonthlyCharges': 114.05, 'TotalCharges': 8468.2, 'InternetService_Fiber optic': 1.0, 'InternetService_No': 0.0, 'OnlineSecurity_No internet service': 0.0, 'OnlineSecurity_Yes': 1.0, 'OnlineBackup_No internet service': 0.0, 'DeviceProtection_No internet service': 0.0, 'TechSupport_No internet service': 0.0, 'TechSupport_Yes': 1.0, 'StreamingTV_No internet se

{'SeniorCitizen': 0.0, 'Partner': 0.0, 'Dependents': 0.0, 'tenure': 11.0, 'PaperlessBilling': 1.0, 'MonthlyCharges': 41.6, 'TotalCharges': 470.6, 'InternetService_Fiber optic': 0.0, 'InternetService_No': 0.0, 'OnlineSecurity_No internet service': 0.0, 'OnlineSecurity_Yes': 0.0, 'OnlineBackup_No internet service': 0.0, 'DeviceProtection_No internet service': 0.0, 'TechSupport_No internet service': 0.0, 'TechSupport_Yes': 0.0, 'StreamingTV_No internet service': 0.0, 'StreamingMovies_No internet service': 0.0, 'Contract_One year': 1.0, 'Contract_Two year': 0.0, 'PaymentMethod_Credit card (automatic)': 0.0, 'PaymentMethod_Electronic check': 1.0}
prediction:  0
--------------------
{'SeniorCitizen': 0.0, 'Partner': 1.0, 'Dependents': 0.0, 'tenure': 29.0, 'PaperlessBilling': 1.0, 'MonthlyCharges': 19.55, 'TotalCharges': 521.8, 'InternetService_Fiber optic': 0.0, 'InternetService_No': 1.0, 'OnlineSecurity_No internet service': 1.0, 'OnlineSecurity_Yes': 0.0, 'OnlineBackup_No internet service'

In [20]:
y_pred_dt

array([0, 1, 0, ..., 0, 0, 0], shape=(1409,))